In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import seaborn.objects as so
from PIL import Image
import numpy as np
from wordcloud import WordCloud, STOPWORDS, ImageColorGenerator
import icd10
import hvplot.pandas
import holoviews as hv
from bokeh.plotting import show
from pycirclize import Circos
from pycirclize.utils import load_example_tree_file
from workspace_dani.data_preprocessing.utils import data_handling

hv.extension("bokeh")


In [ ]:
project_root_path = Path.cwd().parents[2]  #/ Path("data_preprocessing") / Path("utils")

dataset_path = project_root_path / Path("intermediary_data") / Path("dataset") / Path("build") / Path(
    "2024-03-26") / Path("build_dataset_cats.parquet")
dataset_path

In [ ]:
df_data = data_handling.load_parquet_table(parquet_path=dataset_path, debug_mode=False)
df_data

## Cardiogenic Shock in apacheadmissiondx

In [ ]:
admission_dx: pd.Series = df_data["apacheadmissiondx"]

In [ ]:
admission_dx_sepsis: pd.Series = admission_dx[admission_dx.str.contains("Sepsis", case=False)]

In [ ]:
admission_dx_sepsis

### Searching broadly for expressions that contain the first letters of cardiogenic shock

In [ ]:
admission_dx_card: pd.Series = admission_dx[admission_dx.str.contains("Card", case=False)]


In [ ]:
admission_dx_card

In [ ]:
admission_dx_card.unique()

### Cardiogenic Shock is represented as "shock, cardiogenic"

In [ ]:
admission_dx_cardiogenic_shock = admission_dx[admission_dx.str.contains("shock, cardiogenic", case=False)]
print(
    f"{len(admission_dx_cardiogenic_shock)} \n Number of admit diagnosis cardiogenic shock in all patients with a valid APACHE score ")

#### Cardiogenic shock might be shadowed by heart attack, since many cardiogenic shocks are a results of a heart attack (Source: https://www.ucsfhealth.org/conditions/cardiogenic-shock). This could be the reason why there are so few admission diagnoses since, heart attack symptoms will most likely outweigh cardiogenic shock symptoms upon admission.


### Let's investigate how many cardiogenic shock admissions are left when we consider our dataset constraints: Which are right now: Only Patient that are admitted to the Cardiac ICU and are either direct admit or emergency department admits

In [ ]:
admission_source_ids_path = project_root_path / Path(
    "intermediary_data/patient_unit_stay_ids/exclusion/admit_source_unit_type/excluded_ids_admit_source_unit_type.parquet")
admission_source_ids_path

In [ ]:
admission_exclusion_ids_df: pd.DataFrame = data_handling.load_parquet_table(parquet_path=admission_source_ids_path, debug_mode=False)

In [ ]:
admission_exclusion_ids_df


In [ ]:
admission_dx_cardiogenic_shock_included = admission_dx_cardiogenic_shock[
    ~admission_dx_cardiogenic_shock.index.isin(admission_exclusion_ids_df["Excluded ID"])]

print(
    f"{len(admission_dx_cardiogenic_shock_included)} \n Number of admit diagnosis cardiogenic shock in all patients with said cohort constraints APACHE score ")


The difference is only one patient

## Let's investigate the number of cardiogenic shock diagnosis from the diagnosis table

In [ ]:
diagnosis_parquet_path: Path = project_root_path / Path(
    "intermediary_data/data_cleanup/basic_cleanup/2024-03-26/diagnosis.parquet")

In [ ]:
df_diagnosis_table = data_handling.load_parquet_table(parquet_path=diagnosis_parquet_path, debug_mode=False)

In [ ]:
df_diagnosis_table

We will search for the cardiogenic shock in the column "diagnosisstring" and via the ICD-10 code in the column "icd9code", this column contains also the ICD-10 codes even though its name only mentions ICD-9. (The codes with letters are indicative of ICD-10!)

In [ ]:
stay_diagnosis: pd.Series = df_diagnosis_table[["patientunitstayid", "diagnosisstring", "icd9code"]]

_we will consider the ICD-10 code for cardiogenic shock "R57.0" (Source: https://gesund.bund.de/icd-code-suche/r57-0) and the search for the "shock, cardiogenic"_

In [ ]:
shock_cs_icd_code_condition: pd.Series = stay_diagnosis["icd9code"].astype(str).str.contains("r57.0", case=False)

In [ ]:
dx_name_condition: pd.Series = stay_diagnosis["diagnosisstring"].str.contains("shock, cardiogenic", case=False)

In [ ]:
stays_cardiogenic_shock: pd.DataFrame = stay_diagnosis[shock_cs_icd_code_condition | dx_name_condition]

In [ ]:
len(stays_cardiogenic_shock)

### This is a solid amount of diagnosis related to cardiogenic shock, however, this is for the entire subset of patients with a valid APACHE prediction. We will narrow it down to our cohort criteria.


In [ ]:
stays_cardiogenic_shock_included = stays_cardiogenic_shock[
    ~stays_cardiogenic_shock["patientunitstayid"].isin(admission_exclusion_ids_df["Excluded ID"])]


In [ ]:
count_cs_cohort: int = len(stays_cardiogenic_shock_included["patientunitstayid"].unique())
count_cs_cohort

### Our inclusion criteria aren't thinning it out too much, however, we now have to check if those patient unit stay ids are present in our cohort

In [ ]:
df_cohort: pd.DataFrame = df_data[
    ~df_data["patientunitstayid"].isin(admission_exclusion_ids_df["Excluded ID"])]


In [ ]:
len(df_cohort)

Our cohort which consists only of patients with Admitsource=Direct | ED & Unittype=CICU and a valid APACHE prediction, results in only 5322 patient stays that can be considered! This is a substantial reduction in sample number!

In [ ]:
df_cohort_cs = df_data[df_data["patientunitstayid"].isin(stays_cardiogenic_shock_included["patientunitstayid"])]

In [ ]:
len(df_cohort_cs)

**Only 129 patient unit stays included the criteria we discussed in our last meeting, i think its safe to assume that our criteria aren't feasible!**

Let's take a look at our diagnosis in said cohort

In [ ]:
diagnosis_cohort: pd.DataFrame = stay_diagnosis[
    stay_diagnosis["patientunitstayid"].isin(df_cohort["patientunitstayid"])]

In [ ]:
diagnosis_cohort_frequency: dict = diagnosis_cohort["diagnosisstring"].value_counts().to_dict()
diagnosis_cohort_short: dict = {"|".join(k.split(" | ")[-2:]): v for k, v in diagnosis_cohort_frequency.items()}

In [ ]:
diagnosis_stopwords = {"|", " | "}

In [ ]:
diagnosis_cohort_icd_frequency: dict = diagnosis_cohort["icd9code"].value_counts().to_dict()
diagnosis_cohort_icd_short: dict = {r" | ".join(k.split(", ")[-1:]): v for k, v in
                                    diagnosis_cohort_icd_frequency.items()}


In [ ]:
mask_path: Path = project_root_path / Path("workspace_dani/visualizations/assets/Wordcloud ICU Mask.png")
# mask = np.ma.make_mask(np.array(Image.open(mask_path)))
mask = np.array(Image.open(mask_path))


In [ ]:
wordcloud = WordCloud(stopwords=diagnosis_stopwords, background_color="white", max_words=200, width=1700,
                      height=1000, min_font_size=4, relative_scaling=0, contour_width=2,
                      mask=mask).generate_from_frequencies(diagnosis_cohort_short)

In [ ]:
plt.imshow(wordcloud)
plt.axis("off")

In [ ]:
wordcloud.to_file(project_root_path / Path("plot_outputs/wordclouds/johannes_contraints_dx_wordcloud.png"))

Interestingly enough, sepsis is popping up more often than cardiogenic shock!

In [ ]:
df_diagnosis_short: pd.DataFrame = pd.Series(data=diagnosis_cohort_short).reset_index()
df_diagnosis_short.columns = ["diagnosis_short", "frequency"]
df_diagnosis_short = df_diagnosis_short.sort_values(by="frequency", ascending=False)
df_diagnosis_short

In [ ]:
# so.Plot(data=df_diagnosis_short[:20], x="diagnosis_short", y="frequency").add(so.Bar(), orient="x").label(rotation=90)
df_diagnosis_short[:30].hvplot.barh(x="diagnosis_short", y="frequency", width=1640, height=540)


"Cardiovascular | Shock/ hypotension|hypotension" might be an alias for cardiogenic shock with 1522 occurrences.
The diagnosis "Cardiovascular | Shock/hypotension|sepsis" is also quiet common 1220.


In [ ]:
from bigtree import list_to_tree, tree_to_newick

diagnosis_cohort_list: list = diagnosis_cohort["diagnosisstring"].sort_values(
    ascending=True).drop_duplicates().to_list()
diagnosis_cohort_list = [item.replace("|", "/") for item in diagnosis_cohort_list]
diagnosis_cohort_with_common_node = ["".join(["diagnosis/", item]) for item in diagnosis_cohort_list]

In [ ]:
from loguru import logger

with logger.catch():
    root = list_to_tree(diagnosis_cohort_with_common_node)

### Comparison between the diagnosis (cardiogenic shock, sepsis) in the patient cohort johannes (admitsource & unittype) and all the patient with a valid APACHE score

#### Sepsis Cases in the entire considered patient population

In [ ]:
diagnosis_all: pd.DataFrame = df_diagnosis_table

diagnosis_all_frequency: dict = diagnosis_all["diagnosisstring"].value_counts().to_dict()
diagnosis_all_short: dict = {"|".join(k.split(" | ")[-2:]): v for k, v in diagnosis_all_frequency.items()}
diagnosis_all_icd_frequency: dict = diagnosis_all["icd9code"].value_counts().to_dict()
diagnosis_all_icd_short: dict = {r" | ".join(k.split(", ")[-1:]): v for k, v in diagnosis_all_icd_frequency.items()}


In [ ]:
count_cs_all: int = len(stays_cardiogenic_shock["patientunitstayid"].unique())
count_cs_all

In [ ]:
sepsis_icd_code_condition: pd.Series = stay_diagnosis["icd9code"].astype(str).str.contains("A41", case=False)
sepsis_dx_name_condition: pd.Series = stay_diagnosis["diagnosisstring"].str.contains("sepsis", case=False)
stays_sepsis_all: pd.DataFrame = stay_diagnosis[sepsis_icd_code_condition | sepsis_dx_name_condition]


Note: The number of sepsis-related diagnosis is also much higher since patients might suffer multiple times from symptoms that are similiar to sepsis!
-> Determining the number of patients that suffered from sepsis!

In [ ]:
count_sepsis_all: int = len(stays_sepsis_all["patientunitstayid"].unique())
count_sepsis_all


**The number of patients that suffered from sepsis related diagnosis is almost 20x !**


#### Sepsis related diagnosis in the admit and unittype cohort

In [ ]:
stays_sepsis_included: pd.DataFrame = stays_sepsis_all[
    ~stays_sepsis_all["patientunitstayid"].isin(admission_exclusion_ids_df["Excluded ID"])]


In [ ]:
count_sepsis_cohort = len(stays_sepsis_included["patientunitstayid"].unique())
count_sepsis_cohort

Many to many comparison dataframe:
            condition | condition
            sepsis      cs
cohort      729       | 129
all         19013     | 1427 


In [ ]:
conditions_all: dict = {"Sepsis Diagnosis": count_sepsis_all, "Cardiogenic Shock Diagnosis": count_cs_all}
conditions_cohort: dict = {"Sepsis Diagnosis": count_sepsis_cohort, "Cardiogenic Shock Diagnosis": count_cs_cohort}
data = [conditions_cohort, conditions_all]
df_diagnosis_per_population: pd.DataFrame = pd.DataFrame(data=data, index=["Cohort", "Population"],
                                                         columns=["Sepsis Diagnosis", "Cardiogenic Shock Diagnosis"])

In [ ]:
from matplotlib.ticker import EngFormatter

formatter = EngFormatter()

In [ ]:

circos = Circos.initialize_from_matrix(
    df_diagnosis_per_population.T,
    space=20,
    ticks_interval=1000,  # Ticks every 50
    r_lim=(85, 100),  # Radius limits for sectors
    cmap="tab10",  # Use a built-in color map to get better looking colour code
    label_kws=dict(r=125, size=18, color="black"),  # Font of the sector labels
    ticks_kws=dict(label_size=12, label_orientation="vertical", tick_length=3, label_formatter=formatter),
    link_kws=dict(allow_twist=True, direction=0, color="white", lw=0.1, alpha=0.2),  # Style of the links+
)

circle = circos.plotfig(dpi=100)

chord_chart_save_path = project_root_path / Path("plot_outputs/chord_charts/sepsis_vs_cs_admit_source_unittype.png")
circos.savefig(savefile=chord_chart_save_path, dpi=300)


# Taking a look at the distribution of cardiogenic shock and sepsis across ICU types


### Sepsis per ICU type

In [ ]:
sepsis_patient_ids = stays_sepsis_all[stays_sepsis_all["patientunitstayid"].isin(df_data["patientunitstayid"])][
    "patientunitstayid"]

In [ ]:
sepsis_patient_ids.unique()

In [ ]:
df_data_sepsis = df_data[df_data["patientunitstayid"].isin(sepsis_patient_ids)]

In [ ]:
df_sepsis_unit_type = pd.DataFrame(data=df_data_sepsis["unittype"].value_counts()).reset_index()
#df_sepsis_unit_type.columns = ["unittype", "frequency"]
df_sepsis_unit_type["unittype"] = df_sepsis_unit_type["unittype"].astype("category")

In [ ]:
df_sepsis_unit_type.hvplot.barh(x="unittype", y="count", title="Sepsis per ICU Type")

In [ ]:
df_sepsis_unit_type["count"].sum()

### Cardiogenic Shock per ICU Type

In [ ]:
cs_patient_ids = \
    stays_cardiogenic_shock[stays_cardiogenic_shock["patientunitstayid"].isin(df_data["patientunitstayid"])][
        "patientunitstayid"]
cs_patient_ids.unique()


In [ ]:
df_data_cs = df_data[df_data["patientunitstayid"].isin(cs_patient_ids)]

In [ ]:


df_cs_unit_type = pd.DataFrame(data=df_data_cs["unittype"].value_counts()).reset_index()
#df_sepsis_unit_type.columns = ["unittype", "frequency"]
df_cs_unit_type["unittype"] = df_cs_unit_type["unittype"].astype("category")
df_cs_unit_type.hvplot.barh(x="unittype", y="count", title="Cardiogenic Shock per ICU Type")


### Sepsis vs CS with regard to the ICU Type

In [ ]:
fig_pie, axs_pie = plt.subplots(nrows=1, ncols=2)

axs_pie[0].pie(df_sepsis_unit_type["count"], labels=df_sepsis_unit_type["unittype"],
               explode=[0.2 for element in df_sepsis_unit_type["unittype"]])
axs_pie[0].set_title("Sepsis vs. ICU Type")

axs_pie[1].pie(df_cs_unit_type["count"], labels=df_cs_unit_type["unittype"],
               explode=[0.2 for element in df_cs_unit_type["unittype"]])
axs_pie[1].set_title("Cardiogenic Shock vs. ICU Type")
plt.tight_layout()

In [ ]:
color_palette = sns.color_palette("inferno", 8)
unit_types = df_sepsis_unit_type["unittype"].to_list()

label_color = {unit_types[idx]: val for idx, val in enumerate(color_palette)}


In [ ]:
import palettable

fig_donut, ax_donut = plt.subplots(figsize=(10, 6), subplot_kw=dict(aspect="equal"))

wedge_data = df_sepsis_unit_type["count"].to_list()

color_palette = sns.color_palette("inferno", 8)
color_palette_cb = palettable.tableau.ColorBlind_10.mpl_colors

wedges, texts = ax_donut.pie(wedge_data, wedgeprops=dict(width=0.5), startangle=-40, colors=color_palette)

bbox_props = dict(boxstyle="square,pad=0.3", fc="w", ec="k", lw=1)

# kw = dict(bbox=bbox_props, zorder=0, va="center")
kw = dict()

for i, p in enumerate(wedges):
    ang = (p.theta2 - p.theta1) / 2. + p.theta1
    y = np.sin(np.deg2rad(ang))
    x = np.cos(np.deg2rad(ang))

    horizontalalignment = {-1: "right", 1: "left"}[int(np.sign(x))]

    x_text_mod = 1
    y_text_mod = 1.2

    x_text = x_text_mod * float(np.sign(x))
    y_text = y_text_mod * y

    label = df_sepsis_unit_type["unittype"][i]
    value = df_sepsis_unit_type["count"][i]

    # var arrows
    ax_donut.annotate(f"{label} ({value})", xy=(x, y), xytext=(x_text, y_text),
                      horizontalalignment=horizontalalignment,
                      annotation_clip=False,
                      size=15,
                      arrowprops=dict(
                          arrowstyle="->",
                          facecolor='black',
                          lw=2,
                          connectionstyle=f"angle,angleA=0,angleB={ang}"),
                      **kw)
plt.title("Sepsis Diagnoses across ICU Types", size=18)
plt.tight_layout()
plt.show()

In [ ]:
fig_donut, ax_donut = plt.subplots(figsize=(10, 6), subplot_kw=dict(aspect="equal"))

wedge_data = df_cs_unit_type["count"].to_list()

wedges, texts = ax_donut.pie(wedge_data, wedgeprops=dict(width=0.5), startangle=-40, colors=color_palette)

bbox_props = dict(boxstyle="square,pad=0.3", fc="w", ec="k", lw=1)

# kw = dict(bbox=bbox_props, zorder=0, va="center")
kw = dict()

for i, p in enumerate(wedges):
    ang = (p.theta2 - p.theta1) / 2. + p.theta1
    y = np.sin(np.deg2rad(ang))
    x = np.cos(np.deg2rad(ang))

    horizontalalignment = {-1: "right", 1: "left"}[int(np.sign(x))]

    x_text_mod = 1.3
    y_text_mod = 1.2

    x_text = x_text_mod * float(np.sign(x))
    y_text = y_text_mod * y

    label = df_cs_unit_type["unittype"][i]
    value = df_cs_unit_type["count"][i]

    # var arrows
    ax_donut.annotate(f"{label} ({value})", xy=(x, y), xytext=(x_text, y_text),
                      horizontalalignment=horizontalalignment,
                      annotation_clip=False,
                      size=15,
                      arrowprops=dict(
                          arrowstyle="->",
                          facecolor='black',
                          lw=2,
                          connectionstyle=f"angle,angleA=0,angleB={ang}"),
                      **kw)
    
    
plt.title("Cardiogenic Shock Diagnoses across ICU Types", size=18)
plt.tight_layout()
plt.show()


# Investigating different the other types of shock with ICD-10 code R57.x
These ICD codes contain shock of types, cardiogenic (R57.0), hypovolaemic (R57.1), septic (R57.2), misc (including endotoxic) (57.8), unspecified (R57.9) after ICD-10-GM-2024

In [ ]:
shock_hypoval_icd_code_condition: pd.Series = stay_diagnosis["icd9code"].astype(str).str.contains("R57.1", case=False)
shock_septic_icd_code_condition: pd.Series = stay_diagnosis["icd9code"].astype(str).str.contains("R57.2", case=False)
shock_misc_icd_code_condition: pd.Series = stay_diagnosis["icd9code"].astype(str).str.contains("R57.8", case=False)
shock_nonspec_icd_code_condition: pd.Series = stay_diagnosis["icd9code"].astype(str).str.contains("R57.9", case=False)


shock_dx_name_condition: pd.Series = stay_diagnosis["diagnosisstring"].str.contains("shock", case=False)


all_shock_conditions = shock_dx_name_condition | shock_cs_icd_code_condition | shock_hypoval_icd_code_condition | shock_septic_icd_code_condition | shock_misc_icd_code_condition | shock_nonspec_icd_code_condition

stays_shock_all: pd.DataFrame = stay_diagnosis[all_shock_conditions]



### Number of unique patients that suffered from shock at some point:

In [ ]:
stay_diagnosis["patientunitstayid"].unique()

In [ ]:
stays_shock_all["patientunitstayid"].unique()

### Physiological Signals of Sepsis and Cardiogenic Shock Patients

In [ ]:
vitalperiodic_parquet_path: Path = project_root_path / Path(
    "intermediary_data/data_cleanup/basic_cleanup/2024-03-26/vital_periodic.parquet")

df_vitalperiodic = data_handling.load_parquet_table(parquet_path=vitalperiodic_parquet_path, debug_mode=False)

In [ ]:
df_vitalperiodic_sepsis = df_vitalperiodic[df_vitalperiodic["patientunitstayid"].isin(sepsis_patient_ids)]

In [ ]:
df_vitalperiodic_sepsis.describe()

In [ ]:
time_quantile_sepsis = df_vitalperiodic_sepsis["observationoffset"].quantile(q=[0.40, 0.60])

In [ ]:
time_quantile_sepsis_lower = time_quantile_sepsis.iloc[0]
time_quantile_sepsis_upper = time_quantile_sepsis.iloc[1]



In [ ]:
df_vitalperiodic_sepsis_time_valid = df_vitalperiodic_sepsis[df_vitalperiodic_sepsis["observationoffset"].between(time_quantile_sepsis_lower, time_quantile_sepsis_upper)]

In [ ]:

sns.jointplot(x=df_vitalperiodic_sepsis_time_valid["observationoffset"], y=df_vitalperiodic_sepsis_time_valid["heartrate"], ylim=[50,150], kind="hex", color="#4CB391")
